In [ ]:
import uproot
import awkward as ak

import pandas as pd

import matplotlib.pylab as plt
import numpy as np

import hist
from hist import Hist

import time

import myPIDselector

import math

import os

data = ak.from_parquet("Background_and_signal_SP_modes_All_runs.parquet")

plt.close()

def calculate_bits_for_PID_selector(trkidx, trk_selector_map, verbose=0):
    
    bits = None

    # If there is no trk index passed in, just calculate the bits for
    # all of the tracks
    if trkidx is None:
        trkidx = ak.local_index(trk_selector_map)

    # Grab the tracks that map on to the particle/collection we are interested in 
    subset_of_trk_selector_map = trk_selector_map[trkidx]
    if verbose:
        print("values in the subset of the trk selector map")
        print(subset_of_trk_selector_map)
        print()
        
    # We need this to convert the numbers in the selector to binary
    binary_repr_vec = np.vectorize(np.binary_repr)

    # Grab the number of entries in each so we can unflatten this later
    counts = ak.num(subset_of_trk_selector_map)
    
    # Now get the binary representation (as a string) for the flattened subset
    binrep = binary_repr_vec(ak.flatten(subset_of_trk_selector_map), width=32)

    if verbose:
        print("binary representation of selector map")
        print(binrep)
        print()

    # Convert the string to integers
    tempbits = np.array(binrep).astype(int)
    bits = ak.unflatten(tempbits,counts)

    if verbose:
        print("flattened integer representation of selectors as binary (int)")
        print(tempbits)
        print()
        print("unflattened integer representation of selectors as binary (int)")
        print(bits)
        print()

    return bits
def mask_PID_selection(bits, selector, pid_map_object):

    bit_to_look_for = pid_map_object.selectors.index(selector)

    place = int(math.pow(10,bit_to_look_for))
    #print(place)

    mask = bits // place % 10

    mask_bool = ak.values_astype(mask,bool)

    return mask_bool
def get_info_for_PID_masks(ak_arr, verbosity=0):

    # Proton and pion information from the Lambda decay
    # These are the index of the proton (d1) and pion (d2) in those lists
    L1d1idx = ak_arr.Lambda0d1Idx[ak_arr.Bd1Idx]
    L1d2idx = ak_arr.Lambda0d2Idx[ak_arr.Bd1Idx]

    L2d1idx = ak_arr.Lambda0d1Idx[ak_arr.Bd2Idx]
    L2d2idx = ak_arr.Lambda0d2Idx[ak_arr.Bd2Idx]


    
     
    #d1lund = ak_arr['Lambda0d1Lund']
    #d2lund = ak_arr['Lambda0d2Lund']
    
    #Bd2idx = ak_arr['Bd2Idx']
    #Bd2lund = ak_arr['Bd2Lund']
    
    trkidx_proton = data['pTrkIdx']
    trk_selector_map_proton = data['pSelectorsMap']
    
    trkidx_pion = data['piTrkIdx']
    trk_selector_map_pion = data['piSelectorsMap']

    indices_and_maps = {}
    indices_and_maps['L1d1idx'] = L1d1idx
    indices_and_maps['L1d2idx'] = L1d2idx
    indices_and_maps['L2d1idx'] = L2d1idx
    indices_and_maps['L2d2idx'] = L2d2idx
    #indices_and_maps['Bd2idx'] = Bd2idx

    indices_and_maps['trkidx_proton'] = trkidx_proton
    indices_and_maps['trk_selector_map_proton'] = trk_selector_map_proton

    indices_and_maps['trkidx_pion'] = trkidx_pion
    indices_and_maps['trk_selector_map_pion'] = trk_selector_map_pion

    return indices_and_maps
def PID_masks(indices_and_maps, \
              lam1p_selector,\
              lam1pi_selector, \
              lam2p_selector,\
              lam2pi_selector, \
              lamHip_selector, \
              lamHipi_selector, \
             verbosity=0):
                #change Bpselector
    L1d1idx = indices_and_maps['L1d1idx']
    L1d2idx = indices_and_maps['L1d2idx']
    L2d1idx = indices_and_maps['L2d1idx']
    L2d2idx = indices_and_maps['L2d2idx']
    #Bd2idx = indices_and_maps['Bd2idx']

    trkidx_proton = indices_and_maps['trkidx_proton']
    trk_selector_map_proton = indices_and_maps['trk_selector_map_proton']

    trkidx_pion = indices_and_maps['trkidx_pion']
    trk_selector_map_pion = indices_and_maps['trk_selector_map_pion']
    
    # Proton
    pbits = calculate_bits_for_PID_selector(trkidx_proton, trk_selector_map_proton, verbose=verbosity)
    # Pion
    pibits = calculate_bits_for_PID_selector(trkidx_pion, trk_selector_map_pion, verbose=verbosity)
    
    
    #selector_proton = 'TightKMProtonSelection'
    #selector_pion = 'TightKMPionMicroSelection'
    #print(f"Now trying to create a mask with {selector_proton}")
    #print(f"Now trying to create a mask with {selector_pion}")
    
    
    mask_bool_protonL1 = mask_PID_selection(pbits[L1d1idx], lam1p_selector, pps)
        
    mask_bool_pionL1 = mask_PID_selection(pibits[L1d2idx], lam1pi_selector, pips)

    mask_bool_protonL2 = mask_PID_selection(pbits[L2d1idx], lam2p_selector, pps)
        
    mask_bool_pionL2 = mask_PID_selection(pibits[L2d2idx], lam2pi_selector, pips)

    mask_book_protonHiL1 = mask_PID_selection(pbits[L1d1idx], lamHip_selector, pps)
    mask_book_protonHiL2 = mask_PID_selection(pbits[L2d1idx], lamHip_selector, pps)
    mask_book_pionHiL1 = mask_PID_selection(pibits[L1d2idx], lamHipi_selector, pips)
    mask_book_pionHiL2 = mask_PID_selection(pibits[L2d2idx], lamHipi_selector, pips)
    

    return mask_bool_protonL1, mask_bool_pionL1,mask_bool_protonL2, mask_bool_pionL2, mask_book_protonHiL1,mask_book_protonHiL2,mask_book_pionHiL1,mask_book_pionHiL2


pps = myPIDselector.PIDselector("p")
pips = myPIDselector.PIDselector("pi")


fomD = {}
x=0
xL = []
fomL = []
sigL = []
bkgeL = []
bkgtL = []
selL =[]
protonSD = {0:"SuperLooseKMProtonSelection", 1:"VeryLooseKMProtonSelection",2:"LooseKMProtonSelection",3:"TightKMProtonSelection", 4:"VeryTightKMProtonSelection", 5:"SuperTightKMProtonSelection"}
pionSD = {0:"SuperLooseKMPionMicroSelection", 1:"VeryLooseKMPionMicroSelection",2:"LooseKMPionMicroSelection",3:"TightKMPionMicroSelection", 4:"VeryTightKMPionMicroSelection", 5:"SuperTightKMPionMicroSelection"}
for i in range(6):
    myPMask1 = protonSD[i]
    for j in range(6):
        myPiMask1 = pionSD[j]
        for p in range(i,6):
            myPMask = protonSD[p]
            for q in range(j,6):
                myPiMask = pionSD[q]
                x+=1
                indices_and_maps = get_info_for_PID_masks(data, verbosity=0)

                mask_bool_protonL1, mask_bool_pionL1,mask_bool_protonL2, mask_bool_pionL2, mask_book_protonHiL1, mask_book_protonHiL2, mask_book_pionHiL1, mask_book_pionHiL2 = PID_masks(indices_and_maps, \
                                    lam1p_selector=myPMask1, \
                                    lam1pi_selector=myPiMask1, \
                                    lam2p_selector=myPMask1, \
                                    lam2pi_selector=myPiMask1, \
                                    lamHip_selector = myPMask, \
                                    lamHipi_selector = myPiMask, \
                                    verbosity=0)
                sigDict = {'998':0,'1005':0,'-999':0}
                for valu in ['998','1005','-999']:
                    for lengthCut in [1]:
                        valid = []
                        inValid = []
                        passCount = 0
                        valCount = 0
                        inValCount=0
                        emptyCount=0
                        spmask = (data['spmode']==valu)

                        mes = data['BpostFitMes']
                        de = data['BpostFitDeltaE']

                        meslo = 5.27
                        meshi = 5.3
                        DeltaElo = -0.07
                        DeltaEhi = 0.07

                        mask_pid = mask_bool_pionL1 & mask_bool_protonL1 & mask_bool_pionL2 & mask_bool_protonL2 & (mask_book_protonHiL1 | mask_book_protonHiL2) & (mask_book_pionHiL1 | mask_book_pionHiL2)
                        mask_mesdelE = (mes>meslo) & (mes<meshi) & (de>DeltaElo) & (de<DeltaEhi)
                        mask = mask_mesdelE & mask_pid


                        lamconflsig = data[spmask]['Lambda0postFitFlightSignificance'][mask[spmask]]
                        lamfl_basic = data[spmask]['Lambda0FlightLen'][mask[spmask]]
                        lamuncmass = data[spmask]['Lambda0_unc_Mass'][mask[spmask]]
                        numBary = data[spmask]['nB']

                        lamconflsigUS = data[spmask]['Lambda0postFitFlightSignificance'][mask_mesdelE[spmask]]
                        lamfl_basicUS = data[spmask]['Lambda0FlightLen'][mask_mesdelE[spmask]]
                        lamuncmassUS = data[spmask]['Lambda0_unc_Mass'][mask_mesdelE[spmask]]
                        numBaryUS = data[spmask]['nB']


                        m = lamuncmass
                        mUS = lamuncmassUS

                        #flight significance cut
                        mask_fl = (lamfl_basic>lengthCut)&(lamconflsig>0)
                        m = m[mask_fl]
                        m = m[(numBary==1)]
                        mask_fl_US = (lamfl_basicUS>lengthCut)&(lamconflsigUS>0)
                        mUS = mUS[mask_fl_US]
                        mUS = mUS[(numBaryUS==1)]


                            
                        if valu == '-999':
                            sigDict[valu] = ak.sum(ak.num((m)))/ak.sum(ak.num((mUS)))                    
                        elif valu == '998':
                            sigDict[valu]=(ak.sum(ak.num((mUS))),ak.sum(ak.num((m))))
                        elif valu == '1005':
                            sigDict[valu]=(ak.sum(ak.num((mUS))),ak.sum(ak.num((m))))

                        

                fomD[(myPMask1,myPiMask1,myPMask,myPiMask)] = sigDict['-999']/(np.sqrt(sigDict['1005'][0]*(424.3/868.3)+sigDict['998'][0]*(424.3/1719))+2)
                bkgtL += [(sigDict['1005'][1]*(424.3/868.3)+sigDict['998'][1]*(424.3/1719))]
                bkgeL+=[(sigDict['1005'][1]+sigDict['998'][1])/(sigDict['1005'][0]+sigDict['998'][0])]
                sigL+=[sigDict['-999']]
                selL+=[(myPMask1,myPiMask1,myPMask,myPiMask)]
                
                xL += [x]
                fomL += [fomD[(myPMask1,myPiMask1,myPMask,myPiMask)]]

        #print((myPMask1,myPiMask1,myPMask,myPiMask))

#for index, (key, value) in enumerate(fomD.items()):
    #print(f"FOM         / Index: {index}, Key: {key}, Value: {value}")


sorted_data = list(sorted(fomD.items(), key=lambda item: item[1], reverse=True))

for i in range(10):
    print(f'{i+1} / all spmode / {sorted_data[i]}')
    
#for i in [0,25,45,60,75,95,111,123]:
    #print(f'{i+1} // FOM: {fomL[i]:.3g} / Sig Eff: {sigL[i]:.3g} / Total bkg: {bkgtL[i]:.3g} / Bkg Eff: {bkgeL[i]:.3g}\n---//{selL[i]}//')

indL = [0]
fomInd = [fomL[0]]
sigEInd = [sigL[0]]
bkgtInd = [bkgtL[0]]
bkgeInd = [bkgeL[0]]
for i in range(1,260):
    if fomL[i]>fomL[i-1] and fomL[i]>fomL[i+1]:
        indL+=[i]
        fomInd+=[fomL[i]]
        sigEInd+=[sigL[i]]
        bkgtInd+=[bkgtL[i]]
        bkgeInd+=[bkgeL[i]]
    if i == 126:
        indL+=[i]
        fomInd+=[fomL[i]]
        sigEInd+=[sigL[i]]
        bkgtInd+=[bkgtL[i]]
        bkgeInd+=[bkgeL[i]]



for i in indL:
    print(f'{i+1} // FOM: {fomL[i]:.3g} / Sig Eff: {sigL[i]:.3g} / Total bkg after cut: {bkgtL[i]:.3g} / Bkg Eff: {bkgeL[i]:.3g}\n{selL[i]}')


plt.figure()
plt.plot(xL, fomL, 'g-')
plt.plot(indL, fomInd, 'bo')
plt.xlabel('Pid selector index')
plt.ylabel('Figure of merit')
#plt.plot(xL, fomLO, 'b-')
plt.figure()
plt.plot(xL, sigL, 'b-')
plt.plot(indL, sigEInd, 'go')
plt.xlabel('Pid selector index')
plt.ylabel('Sig eff')
plt.figure()
plt.plot(xL, bkgeL, 'b-')
plt.plot(indL,  bkgeInd, 'go')
plt.xlabel('Pid selector index')
plt.ylabel('bkg eff ((998cut+1005cut)/(998uncut+1005uncut))')
plt.figure()
plt.plot(xL, bkgtL, 'r-')
plt.plot(indL,  bkgtInd, 'bo')
plt.xlabel('Pid selector index')
plt.ylabel('Cut Background (998cut*(424.3/1719)+1005cut*(424.3/868.3))')

plt.show()
